# get model trained on all, ensemble trained on all, ensemble refined

### imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [130]:
# Load config
import sys
import os
from pathlib import Path
from IPython.display import display


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

from model_in_the_loop.utils.hydra_utils import load_config,set_env_vars
cfg = load_config()
set_env_vars(cfg)  # set env variables for repo and data paths


home directory: /gpfs01/euler/User/ssuhai


In [129]:
cfg.model_configs.paths

{'load_model_path': None, 'cache_dir': '${oc.env:OPENRETINA_CACHE_DIRECTORY}', 'data_dir': '${paths.cache_dir}/data/', 'log_dir': '.', 'output_dir': '${hydra:runtime.output_dir}', 'movies_path': 'https://huggingface.co/datasets/open-retina/open-retina/blob/main/euler_lab/hoefling_2024/stimuli/rgc_natstim_18x16_joint_normalized_2024-01-11.zip', 'responses_path': 'https://huggingface.co/datasets/open-retina/open-retina/resolve/main/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.zip'}

In [21]:
from model_in_the_loop.core.dj_wrappers import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper)

from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure,rm_all_experiment_dirs,clear_data_dump_dir
from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata,create_rf_test_dir
from model_in_the_loop.utils.simple_logging import log
from model_in_the_loop.utils.plotting import show_all_rois_plot

### Create processing components (connect them to DB)

In [23]:
# create preprocessor

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore

                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [24]:
dj_table_holder.setup()


[2025-10-03 13:48:14,026][INFO]: Connecting ssuhai@172.25.240.200:3306
[2025-10-03 13:48:14,084][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop
Done reconnecting. Skipping adding new entries from config.


/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/core/dj_wrappers/base.py:293: UserWarning: 
Some DJ tables (like UserInfo) are not empty, skipping adding new entries from config.
Make sure this is wanted. Call clear_tables(`all`) if you want different data in there
  warnings.warn("\nSome DJ tables (like UserInfo) are not empty, skipping adding new entries from config.\nMake sure this is wanted. Call clear_tables(`all`) if you want different data in there")


In [25]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

## get data

### Move files from server to the repo 

In [26]:
# create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)

# copy_rec_files(
#     recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
#     destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
#     full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
# )

### Preprocessing

In [13]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251001/1
		header_name: 20251001_left.ini
		Already present: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 10, 1, 0, 0), 'exp_num': 1}
Found 7 files in 2 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1}
	Skipping field `{'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1}` because it already exists
	Adding field: `{'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1}`


Processes: 100%|██████████| 9/9 [00:07<00:00,  1.15it/s]


In [27]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

No missing fields found, using the last field key.
Field key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1'}


In [15]:
# compute 
preprocessor.add_iteration_roi_mask(field_key=field_key)
preprocessor.add_iteration_rois()
preprocessor.add_iteration_traces()


field_key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1'} 
pres_keys: [{'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1', 'stim_name': 'densenoise', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1', 'stim_name': 'gChirp', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1', 'stim_name': 'mouse_cam', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 1), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter1', 'stim_name': 'movingbar', 'cond2': 'control'}]
No ROI masks found for field: {'experimenter

Returned InteractiveRoiCanvas object. To start GUI, call <enter_object_name>.start_gui().


Processes: 100%|██████████| 428/428 [00:00<00:00, 562.22it/s]


### qualty and RF

In [28]:
quality_type_analysis_wrapper.compute_analysis(
    field_key=field_key)

# filter 
passed_roi_ids_chirp_mb = quality_type_analysis_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
    d_qi_min =cfg.quality_filtering["d_qi_min"],
    qidx_min=cfg.quality_filtering["chirp_qi_min"],
    celltypes=cfg.quality_filtering["celltypes"],
    classifier_confidence=cfg.quality_filtering["classifier_confidence"])
if len(passed_roi_ids_chirp_mb) == 0:
    raise ValueError("No ROIs passed the quality criterion for quality and type.")
print(f"{len(passed_roi_ids_chirp_mb)} ROIs passed quality chirp mb filtering: {passed_roi_ids_chirp_mb}")



Number after filtering rois based on classifier confidence 80.
Number after filtered rois based on celltypes 95.
Number after filtered rois based on Chirp MB QI 90.
Found 66 rois passing the criterion out of 107 rois.                (d_qi_min=0.6, chrip qidx_min=0.35, celltypes=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32], classifier_confidence=0.25)
66 ROIs passed quality chirp mb filtering: [5, 7, 8, 12, 13, 14, 15, 16, 17, 21, 22, 24, 25, 26, 27, 28, 29, 31, 34, 35, 36, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 51, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 67, 68, 71, 72, 73, 74, 75, 78, 80, 81, 82, 84, 86, 89, 91, 92, 93, 94, 97, 99, 101, 102, 103]


/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/classifier/rgc_classifier.py:402: UserWarning: Parallel processing not implemented!
  warnings.warn('Parallel processing not implemented!')


In [29]:
# sta 
sta_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_chirp_mb,)


In [30]:
assert ((dj_table_holder("STA")() & field_key).fetch("roi_id") == passed_roi_ids_chirp_mb).all(), "STA roi_id does not match passed roi_ids from quality and type filtering."

In [99]:
# filter
passed_roi_ids_sta = sta_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
                                                               rf_qidx_min= cfg.quality_filtering["rf_qidx_min"])
if len(passed_roi_ids_sta) == 0:
    raise ValueError("No ROIs passed the quality criterion for STA.")
print(f"{len(passed_roi_ids_sta)} ROIs passed STA filtering with rf_qidx_min={cfg.quality_filtering["rf_qidx_min"]}: {passed_roi_ids_sta}")

if len(passed_roi_ids_sta) < 25:
    print("HAVE AT LEAST 25 ROIS")


51 ROIs passed STA filtering with rf_qidx_min=0.5: [5, 7, 8, 14, 15, 21, 22, 24, 25, 26, 27, 28, 29, 34, 35, 36, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 51, 53, 56, 59, 60, 61, 62, 63, 67, 68, 71, 73, 74, 78, 81, 84, 86, 89, 91, 92, 93, 94, 102, 103]


In [ ]:
# if len(passed_roi_ids_sta) >= 25:
#     print("MORE THAN 25 ROIS, selecting 25 best")
#     top_n_rois_sta,top_n_scores = sta_wrapper.list_top_n_rois_by_qidx(field_key=field_key,
#                                                     n=25,)
#     passed_roi_ids_sta = top_n_rois_sta
#     print(top_n_rois_sta)
#     print(top_n_scores)
#     print(f"Using top 25 rois based on rf_qidx: {passed_roi_ids_sta}")

HAVE AT LEAST 25 ROIS
[45, 47, 74, 86, 59, 36, 71, 62, 24, 41, 25, 26, 46, 84, 60, 7, 40, 38, 29, 89, 63, 51, 8, 94, 14]
[0.905903, 0.896839, 0.888341, 0.872203, 0.870396, 0.863777, 0.858296, 0.857354, 0.843758, 0.839727, 0.83969, 0.835443, 0.819608, 0.805396, 0.800651, 0.796001, 0.784526, 0.781724, 0.769277, 0.755537, 0.754558, 0.753798, 0.742307, 0.739806, 0.736961]
Using top 25 rois based on rf_qidx: [45, 47, 74, 86, 59, 36, 71, 62, 24, 41, 25, 26, 46, 84, 60, 7, 40, 38, 29, 89, 63, 51, 8, 94, 14]


# get wrappers

In [110]:
def get_desired_wrapper(cfg,mode,field_key,roi_id_subset):
    if mode == "single_member_train_full":
        cfg.model_configs.only_train_readout = False
        cfg.model_configs.is_ensemble_model = False
    elif mode == "refine_ensemble":
        cfg.model_configs.only_train_readout = True
        cfg.model_configs.is_ensemble_model = True
        print(cfg.model_configs.paths.load_model_path)
    else:
        raise ValueError(f"Mode {mode} not recognized.")



    random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
    )

    # compute 
    random_seed_mei_wrapper.compute_analysis(
        field_key=field_key,
        roi_id_subset=roi_id_subset,
        )

    return random_seed_mei_wrapper


In [117]:
mei_wrapper_all = get_desired_wrapper(cfg=cfg,mode="single_member_train_full",field_key=field_key,roi_id_subset=passed_roi_ids_sta)

Processes: 100%|██████████| 51/51 [00:00<00:00, 460.42it/s]


	YAML reader installed (version 0.18.6).
	Keras installed (version 3.8.0).
	Tensorflow installed (version 2.16.1).


CascadeSpikes:   0%|          | 0/1 [00:00<?, ?it/s]2025-10-03 19:33:38.683177: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


CascadeSpikes: 100%|██████████| 1/1 [00:20<00:00, 20.74s/it]


Loading HDF5 file contents:   0%|          | 0/2077 [00:00<?, ?item/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_correlation did not improve in the last 10 records. Best score: 0.248. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃       DataLoader 1        ┃       DataLoader 2        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test_correlation      │    0.17994637787342072    │    0.24889300763607025    │    0.5293732285499573     │
│         test_loss         │     632.8604125976562     │     485.1717834472656     │     54.20860290527344     │
└───────────────────────────┴───────────────────────────┴───────────────────────────┴───────────────────────────┘

Filtered neurons based on testset correlation: 51 -> 46
Generating unstable MEIs for neurons (readout idx): [4, 8, 12, 27, 28, 39, 43, 45, 49].
Done with meis in phase unstable.
Start decomposing ...
Decomposing MEIs for neuron (readout idx) 4 ...
changing norm of reconstruction 23.119335174560547 to match original mei norm 29.999780654907227
new reconstruction norm 29.999780654907227
Done reconstructing MEI for neuron (readout idx) 4, seed 111.
changing norm of reconstruction 23.536182403564453 to match original mei norm 29.99953842163086
new reconstruction norm 29.999540328979492
Done reconstructing MEI for neuron (readout idx) 4, seed 222.
Decomposing MEIs for neuron (readout idx) 8 ...
changing norm of reconstruction 23.80482292175293 to match original mei norm 30.0
new reconstruction norm 30.000001907348633
Done reconstructing MEI for neuron (readout idx) 8, seed 111.
changing norm of reconstruction 23.28700065612793 to match original mei norm 29.999996185302734
new reconstruction

In [120]:
mei_wrapper_all.save_all_data_to_dir(mei_wrapper_all.save_dir_parent)

Saved raw session dict to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_200456/session_dict_raw.pkl
Saved neuron data dict to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_200456/neuron_data_dict.pkl
Saved MEI data container to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_200456/mei_data_container.pkl
Saved model state dict to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_200456/model_state_dict.pt
Saved full model to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_200456/model_full.pt
Saved metadata to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_200456/metadata.pkl


In [132]:
1 + 1
mei_wrapper_refine_ensemble = get_desired_wrapper(cfg=cfg,mode="refine_ensemble",field_key=field_key,roi_id_subset=passed_roi_ids_sta)

/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/models/full_ensemble
	YAML reader installed (version 0.18.6).
	Keras installed (version 3.8.0).
	Tensorflow installed (version 2.16.1).


Upsampling natural spikes traces to get final responses.:   0%|          | 0/1 [00:00<?, ?it/s]

Creating movie dataloaders:   0%|          | 0/1 [00:00<?, ?it/s]

Seed set to 2000
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/fabric/loggers/csv_logs.py:268: Experiment logs directory /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/outputs/csv/ exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/outputs/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                        | Params | Mode 
-------------------------------------------------------------------------
0 | core             | SimpleCoreWrapper           | 12.4 K | train
1 | readout          | MultiGau

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_correlation did not improve in the last 10 records. Best score: 0.335. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃       DataLoader 1        ┃       DataLoader 2        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test_correlation      │    0.2797206938266754     │    0.33597108721733093    │     0.549828290939331     │
│         test_loss         │     546.4498901367188     │     422.2274169921875     │    42.709774017333984     │
└───────────────────────────┴───────────────────────────┴───────────────────────────┴───────────────────────────┘

Filtered neurons based on testset correlation: 51 -> 48
Generating unstable MEIs for neurons (readout idx): [4, 12, 20, 25, 27, 29, 39, 42, 45].
Done with meis in phase unstable.
Start decomposing ...
Decomposing MEIs for neuron (readout idx) 4 ...
changing norm of reconstruction 21.638076782226562 to match original mei norm 29.999996185302734
new reconstruction norm 29.999998092651367
Done reconstructing MEI for neuron (readout idx) 4, seed 111.
changing norm of reconstruction 24.243553161621094 to match original mei norm 30.0
new reconstruction norm 30.0
Done reconstructing MEI for neuron (readout idx) 4, seed 222.
Decomposing MEIs for neuron (readout idx) 12 ...
changing norm of reconstruction 21.890901565551758 to match original mei norm 29.999998092651367
new reconstruction norm 29.999998092651367
Done reconstructing MEI for neuron (readout idx) 12, seed 111.
changing norm of reconstruction 22.956857681274414 to match original mei norm 29.999235153198242
new reconstruction norm 29

## visualize with GUI

In [ ]:
from model_in_the_loop.core.gui import ExtendedAutoRoiGui
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,mei_wrapper_all],
    #take_roi_rgba_from_this_analysis=quality_type_analysis_wrapper.name,
    take_roi_rgba_from_this_analysis=mei_wrapper_all.name,
    # JUST VIS
    do_not_compute_only_visualize = True,
    
    field_key=field_key,
    canvas_width=30)

[2025-10-03 19:59:23,395][WARNING]: MySQL server has gone away. Reconnecting to the server.


--- Logging error ---
Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/datajoint/connection.py", line 343, in query
    self._execute_query(cursor, query, args, suppress_warnings)
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/datajoint/connection.py", line 299, in _execute_query
    raise translate_query_error(err, query)
datajoint.errors.LostConnectionError: ('Server connection lost', 'Lost connection to MySQL server during query ([SSL: TLSV1_ALERT_DECODE_ERROR] tlsv1 alert decode error (_ssl.c:2570))')

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
ValueError: I/O operation on closed file.
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  F

Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml
Loaded initial ROI mask with 107 ROIs.


In [119]:
display(gui.start_gui())

In [ ]:
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,mei_wrapper_refine_ensemble],
    #take_roi_rgba_from_this_analysis=quality_type_analysis_wrapper.name,
    take_roi_rgba_from_this_analysis=mei_wrapper_refine_ensemble.name,
    # JUST VIS
    do_not_compute_only_visualize = True,
    
    field_key=field_key,
    canvas_width=30)



Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml
Loaded initial ROI mask with 107 ROIs.


In [135]:
display(gui.start_gui())

In [136]:
mei_wrapper_refine_ensemble.save_all_data_to_dir(mei_wrapper_refine_ensemble.save_dir_parent)

Saved raw session dict to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_212914/session_dict_raw.pkl
Saved neuron data dict to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_212914/neuron_data_dict.pkl
Saved MEI data container to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_212914/mei_data_container.pkl
Saved model state dict to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_212914/model_state_dict.pt
Saved full model to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_212914/model_full.pt
Saved metadata to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/online_computed_data/GCL0_20251003_212914/metadata.pkl


In [138]:
def get_mei_order(wrapper, wrapper_name):
    roi_id2mei_ids, roi_id2info = wrapper.select_subset_of_meis_for_each_roi(
                                            neuron_data_dict =wrapper.neuron_data_dict,
                                            new_session_id =wrapper.new_session_id,
                                            mei_data_container = wrapper.mei_data_container,
                                            readout_idx_wmei2rois = wrapper.readout_idx_wmei2rois,
                                            neuron_idxs_passing_filter = wrapper.neuron_idxs_passing_filter,
                                            n_stimuli_total = 6,
    )
    parent_dir ="/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis"
    # save 
    import pickle 
    with open(os.path.join(parent_dir,f"{wrapper_name}_roi_id2mei_ids.pkl"),"wb") as f:
        pickle.dump(roi_id2mei_ids,f)
    with open(os.path.join(parent_dir,f"{wrapper_name}_roi_id2info.pkl"),"wb") as f:
        pickle.dump(roi_id2info,f)
    return roi_id2mei_ids, roi_id2info



ensemble_roi_id2mei_ids, ensemble_roi_id2info = get_mei_order(mei_wrapper_refine_ensemble,"refine_ensemble")
member_roi_id2mei_ids, member_roi_id2info = get_mei_order(mei_wrapper_all,"single_member_train_full")

In [149]:
def save_strfs():
    roi_id,srf,trf = (dj_table_holder("SplitRF")() & "cond1='iter1'").fetch("roi_id","srf","trf")
    parent_dir ="/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis"
    import pickle 
    with open(os.path.join(parent_dir,"roi_id_srf_trf.pkl"),"wb") as f:
        pickle.dump((roi_id,srf,trf),f)

save_strfs()

In [151]:
# dj_table_holder("CelltypeAssignment")()

In [ ]:
def save_celltype():
    parent_dir ="/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis"
    
    

In [ ]:
def get_data_container_fully_trained_ensemble(model):
    import torch
    from model_in_the_loop.utils.stimulus_optimization import generate_opt_stim_for_neuron_list
    from model_in_the_loop.utils.stimulus_optimization import decompose_mei,reconstruct_mei_from_decomposed
    from model_in_the_loop.utils.model_training import BaseCoreRead
    # initialize mei data containter
    mei_data_container_entries = []

    ## generate meis
    for phase in ['unstable', 'stable']:
        print(f"Generating {phase} MEIs for neurons (readout idx): {[idx for idx,stab in idx2stability.items() if stab ==phase ]}.")
        log(f"Generating {phase} MEIs for neurons (readout idx): {[idx for idx,stab in idx2stability.items() if stab == phase ]}.")
        set_model_to_eval_mode = True if phase == 'stable' else False
        neuron_ids_to_analyze = [neuron_id for neuron_id, stability in idx2stability.items() if stability == phase]
        seeds = self.seeds if phase == 'unstable' else [self.seeds[0]] # only one seed for stable meis

        neuron_seed_mei_dict =  generate_opt_stim_for_neuron_list(
                                        model = self.model,
                                        new_session_id = self.new_session_id,
                                        opt_stim_generation_params= self.mei_generation_params,
                                        random_seeds = seeds,
                                        seed_it_func= torch.manual_seed,
                                        neuron_ids_to_analyze = neuron_ids_to_analyze, # NOTE: this will optimize each id individually 
                                        set_model_to_eval_mode = set_model_to_eval_mode, # model in training mode for noisy MEIs
                                        )
        print(f"Done with meis in phase {phase}.")
        print(f"Start decomposing ...")    
        
        ## decompose meis
        device = "cuda"
        for neuron_id,seed_dict in neuron_seed_mei_dict.items():
            print(f"Decomposing MEIs for neuron (readout idx) {neuron_id} ...")
            for seed,mei in seed_dict.items():

                # decompose the MEIs
                temporal_kernels, spatial_kernels, _ = decompose_mei(stimulus = mei.detach().cpu().numpy())
            

                if self.mei_generation_params["reconstruct_mei"]:
                    reconstruction = reconstruct_mei_from_decomposed(
                                temporal_kernels=temporal_kernels,
                                spatial_kernels=spatial_kernels,
                                turn_to_tensor=True)

                    assert reconstruction.shape == mei.shape, "Reconstructed MEI shape does not match original MEI shape."
                    
                    # make reonstruction same norm as mei
                    print(f"changing norm of reconstruction {torch.norm(reconstruction)} to match original mei norm {torch.norm(mei)}")
                    reconstruction = reconstruction / torch.norm(reconstruction) * torch.norm(mei)
                    print(f"new reconstruction norm {torch.norm(reconstruction)}")
                    mei = reconstruction # use the reconstructed MEI for further analysis
                    print(f"Done reconstructing MEI for neuron (readout idx) {neuron_id}, seed {seed}.")
                    # add entry to data container 
                    mei_data_container_entries.append({
                        "readout_idx": neuron_id,
                        "roi_id": self.readout_idx_wmei2rois[neuron_id],
                        "mei_id": f"roi_{self.readout_idx_wmei2rois[neuron_id]}_seed_{seed}",
                        "seed": seed,
                        "mei": mei.detach(),
                        "temporal_kernels": temporal_kernels,
                        "spatial_kernels": spatial_kernels,
                        "stability": phase,
                    })
        

                    

    # make df container from all meis
    self.mei_data_container = pd.DataFrame(mei_data_container_entries)

    print(f"Generating responses for neurons in readout {len(self.neuron_idxs_passing_filter)} to all meis {len(self.mei_data_container)} ...")
    self.get_responses_and_add_to_container(mei_data_container=self.mei_data_container,
                                            model=self.model,
                                            new_session_id=self.new_session_id,
                                            neuron_data_dict=self.neuron_data_dict,
                                            

# Plot 

In [ ]:
def plot_optimized_kernel_and_trf()